In [16]:
# Исправленная версия - добавьте эту ячейку в начало
%pip install d2l==0.17.0
%pip install tensorflow
%pip install matplotlib-inline

import tensorflow as tf
from d2l import tensorflow as d2l
import matplotlib.pyplot as plt
import numpy as np
import time


# Исправление для новой версии IPython
import IPython.display as display
try:
    display.set_matplotlib_formats('svg')
except AttributeError:
    # Для новых версий IPython
    from matplotlib_inline import backend_inline
    backend_inline.set_matplotlib_formats('svg')

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
def simplified_alexnet_224():
    return tf.keras.models.Sequential([
        # Первый сверточный блок - уменьшено количество фильтров
        tf.keras.layers.Conv2D(filters=48, kernel_size=11, strides=4, activation='relu'),
        tf.keras.layers.MaxPool2D(pool_size=3, strides=2),

        # Второй сверточный блок - уменьшено количество фильтров
        tf.keras.layers.Conv2D(filters=128, kernel_size=5, padding='same', activation='relu'),
        tf.keras.layers.MaxPool2D(pool_size=3, strides=2),

        # Третий сверточный блок - уменьшено количество фильтров
        tf.keras.layers.Conv2D(filters=192, kernel_size=3, padding='same', activation='relu'),
        tf.keras.layers.Conv2D(filters=192, kernel_size=3, padding='same', activation='relu'),
        tf.keras.layers.Conv2D(filters=128, kernel_size=3, padding='same', activation='relu'),
        tf.keras.layers.MaxPool2D(pool_size=3, strides=2),

        # Полносвязные слои - значительно уменьшено количество нейронов
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10)
    ])

In [18]:
def alexnet_28x28():
    return tf.keras.models.Sequential([
        # Изменены параметры для маленьких изображений
        tf.keras.layers.Conv2D(filters=32, kernel_size=5, strides=1, padding='same', activation='relu'),
        tf.keras.layers.MaxPool2D(pool_size=2, strides=2),

        tf.keras.layers.Conv2D(filters=64, kernel_size=3, padding='same', activation='relu'),
        tf.keras.layers.MaxPool2D(pool_size=2, strides=2),

        tf.keras.layers.Conv2D(filters=96, kernel_size=3, padding='same', activation='relu'),
        tf.keras.layers.Conv2D(filters=64, kernel_size=3, padding='same', activation='relu'),
        tf.keras.layers.MaxPool2D(pool_size=2, strides=2),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10)
    ])

# Функция для проверки размерностей
def check_output_shapes(model, input_shape):
    X = tf.random.uniform(input_shape)
    print("Проверка размерностей выходов слоев:")
    for layer in model.layers:
        X = layer(X)
        print(f"{layer.__class__.__name__:20} output shape: {X.shape}")

In [19]:
# Исправленная функция для обучения
def train_and_evaluate(model_fn, batch_size, resize=None, model_name=""):
    print(f"\n{'='*50}")
    print(f"Обучение модели: {model_name}")
    print(f"Batch size: {batch_size}, Resize: {resize}")
    print(f"{'='*50}")

    try:
        # Загрузка данных
        train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size, resize=resize)
        
        # Обучение модели - передаем функцию, а не экземпляр
        lr, num_epochs = 0.01, 10
        model = d2l.train_ch6(model_fn, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())
        
        return model
    except Exception as e:
        print(f"Ошибка при обучении: {e}")
        import traceback
        traceback.print_exc()
        return None

# Для проверки архитектуры создаем отдельный экземпляр
print("Упрощенная AlexNet для 224x224")
test_model = simplified_alexnet_224()
test_model.build((None, 224, 224, 1))
print("Архитектура модели:")
test_model.summary()

check_output_shapes(test_model, (1, 224, 224, 1))

# Для обучения передаем функцию, а не экземпляр
trained_model1 = train_and_evaluate(simplified_alexnet_224, batch_size=128, resize=224, model_name="Упрощенная AlexNet 224x224")

Упрощенная AlexNet для 224x224
Архитектура модели:


Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_63 (Conv2D)              │ (None, 54, 54, 48)     │         5,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_39 (MaxPooling2D) │ (None, 26, 26, 48)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_64 (Conv2D)              │ (None, 26, 26, 128)    │       153,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_40 (MaxPooling2D) │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_65 (Conv2D)              │ (None, 12, 12, 192)    │       221,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_66 (Conv2D)              │ (None, 12, 12, 192)    │       331,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_67 (Conv2D)              │ (None, 12, 12, 128)    │       221,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_41 (MaxPooling2D) │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_13 (Flatten)            │ (None, 3200)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 1024)           │     3,277,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,741,994 (18.09 MB)

 Trainable params: 4,741,994 (18.09 MB)

 Non-trainable params: 0 (0.00 B)

Проверка размерностей выходов слоев:
Conv2D               output shape: (1, 54, 54, 48)
MaxPooling2D         output shape: (1, 26, 26, 48)
Conv2D               output shape: (1, 26, 26, 128)
MaxPooling2D         output shape: (1, 12, 12, 128)
Conv2D               output shape: (1, 12, 12, 192)
Conv2D               output shape: (1, 12, 12, 192)
Conv2D               output shape: (1, 12, 12, 128)
MaxPooling2D         output shape: (1, 5, 5, 128)
Flatten              output shape: (1, 3200)
Dense                output shape: (1, 1024)
Dropout              output shape: (1, 1024)
Dense                output shape: (1, 512)
Dropout              output shape: (1, 512)
Dense                output shape: (1, 10)

Обучение модели: Упрощенная AlexNet 224x224
Batch size: 128, Resize: 224
Ошибка при обучении: module 'IPython.display' has no attribute 'set_matplotlib_formats'


Traceback (most recent call last):
  File "C:\Users\ro517\AppData\Local\Temp\ipykernel_8604\1055927410.py", line 14, in train_and_evaluate
    model = d2l.train_ch6(model_fn, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ro517\AppData\Local\Programs\Python\Python312\Lib\site-packages\d2l\tensorflow.py", line 483, in train_ch6
    callback = TrainCallback(net, train_iter, test_iter, num_epochs,
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ro517\AppData\Local\Programs\Python\Python312\Lib\site-packages\d2l\tensorflow.py", line 446, in __init__
    self.animator = d2l.Animator(
                    ^^^^^^^^^^^^^
  File "c:\Users\ro517\AppData\Local\Programs\Python\Python312\Lib\site-packages\d2l\tensorflow.py", line 276, in __init__
    d2l.use_svg_display()
  File "c:\Users\ro517\AppData\Local\Programs\Python\Python312\Lib\site-package

In [20]:
print("\nAlexNet адаптированная для 28x28")
model2 = alexnet_28x28()
model2.build((None, 28, 28, 1))
print("Архитектура модели:")
model2.summary()

check_output_shapes(model2, (1, 28, 28, 1))
trained_model2 = train_and_evaluate(alexnet_28x28, batch_size=128, resize=None, model_name="AlexNet 28x28")


AlexNet адаптированная для 28x28
Архитектура модели:


Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_73 (Conv2D)              │ (None, 28, 28, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_45 (MaxPooling2D) │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_74 (Conv2D)              │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_46 (MaxPooling2D) │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_75 (Conv2D)              │ (None, 7, 7, 96)       │        55,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_76 (Conv2D)              │ (None, 7, 7, 64)       │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_47 (MaxPooling2D) │ (None, 3, 3, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_15 (Flatten)            │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 256)            │       147,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_30 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_31 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 311,978 (1.19 MB)

 Trainable params: 311,978 (1.19 MB)

 Non-trainable params: 0 (0.00 B)

Проверка размерностей выходов слоев:
Conv2D               output shape: (1, 28, 28, 32)
MaxPooling2D         output shape: (1, 14, 14, 32)
Conv2D               output shape: (1, 14, 14, 64)
MaxPooling2D         output shape: (1, 7, 7, 64)
Conv2D               output shape: (1, 7, 7, 96)
Conv2D               output shape: (1, 7, 7, 64)
MaxPooling2D         output shape: (1, 3, 3, 64)
Flatten              output shape: (1, 576)
Dense                output shape: (1, 256)
Dropout              output shape: (1, 256)
Dense                output shape: (1, 128)
Dropout              output shape: (1, 128)
Dense                output shape: (1, 10)

Обучение модели: AlexNet 28x28
Batch size: 128, Resize: None
Ошибка при обучении: module 'IPython.display' has no attribute 'set_matplotlib_formats'


Traceback (most recent call last):
  File "C:\Users\ro517\AppData\Local\Temp\ipykernel_8604\1055927410.py", line 14, in train_and_evaluate
    model = d2l.train_ch6(model_fn, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ro517\AppData\Local\Programs\Python\Python312\Lib\site-packages\d2l\tensorflow.py", line 483, in train_ch6
    callback = TrainCallback(net, train_iter, test_iter, num_epochs,
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ro517\AppData\Local\Programs\Python\Python312\Lib\site-packages\d2l\tensorflow.py", line 446, in __init__
    self.animator = d2l.Animator(
                    ^^^^^^^^^^^^^
  File "c:\Users\ro517\AppData\Local\Programs\Python\Python312\Lib\site-packages\d2l\tensorflow.py", line 276, in __init__
    d2l.use_svg_display()
  File "c:\Users\ro517\AppData\Local\Programs\Python\Python312\Lib\site-package

In [21]:
# Задание 3
print("\n Влияние размера пакета на производительность")
batch_sizes = [64, 128, 256]

for batch_size in batch_sizes:
    print(f"\nТестирование с batch_size = {batch_size}")

    # Создаем новую модель для каждого эксперимента
    model = alexnet_28x28()

    # Мониторинг использования памяти
    print("Использование памяти GPU до обучения:")
    !nvidia-smi --query-gpu=memory.used --format=csv

    train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size, resize=None)

    lr, num_epochs = 0.01, 5  # Уменьшим эпохи для быстрого тестирования
    d2l.train_ch6(model, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())

    print("Использование памяти GPU после обучения:")
    !nvidia-smi --query-gpu=memory.used --format=csv


 Влияние размера пакета на производительность

Тестирование с batch_size = 64
Использование памяти GPU до обучения:
memory.used [MiB]
19 MiB


TypeError: missing a required argument: 'inputs'

In [ ]:
def visualize_model_architecture():
    # Создаем модели
    model_224 = simplified_alexnet_224()
    model_28 = alexnet_28x28()

    model_224.build((None, 224, 224, 1))
    model_28.build((None, 28, 28, 1))

    print("Упрощенная AlexNet (224x224):")
    for i, layer in enumerate(model_224.layers):
        print(f"{i+1:2d}. {layer.__class__.__name__:15} {layer.output_shape}")

    print("\nAlexNet адаптированная (28x28):")
    for i, layer in enumerate(model_28.layers):
        print(f"{i+1:2d}. {layer.__class__.__name__:15} {layer.output_shape}")

visualize_model_architecture()